In [ ]:
import pandas as pd
import torch
import lightgbm as lgb
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('rlc_dataset.csv')

In [ ]:
X = df[['R', 'L', 'C', 'U']]
y = df['I_max']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_data = lgb.Dataset(X_train,label=y_train)
valid_data = lgb.Dataset(X_test,label=y_test,reference=train_data)
params = {
    "objective":"regression",
    "metric":"rmse",
    "learning_rate":0.1,
    "num_leaves":40,
    "verbose":-1
}

lgb_train = lgb.train(
    params= params,
    train_set=train_data,
    num_boost_round=5000,
    valid_sets=[train_data, valid_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50),lgb.log_evaluation(period=100)]
)
y_pred = lgb_train.predict(X_test)
print(f"R2 on rlc dataset: {r2_score(y_test, y_pred):.10f}")